# Understanding IFC Files: Building Code Compliance Exercises

## What is IFC?

**IFC (Industry Foundation Classes)** is a standard data format for building information. It stores geometric, spatial, and semantic information about buildings in a structured way. Think of it as a database format for architectural and engineering data.

Industries use IFC to:
- Share building models across different software
- Extract specific information (spaces, materials, systems)
- Validate designs against building codes
- Run simulations and analysis

## Resources

Before starting, explore these resources to understand IFC better:

- **IFC Knowledge Base**: https://notebooklm.google.com/notebook/0925c2a1-519b-40a8-aca4-1e832d219f3c
- **BuildingSmart (IFC Standard)**: https://www.buildingsmart.org/standards/bsi-standards/industry-foundation-classes/
- **IfcOpenShell Python Docs**: https://docs.ifcopenshell.org/ifcopenshell-python.html
- **Catalan Building Code Reference**: https://notebooklm.google.com/notebook/216b245f-0fc1-4063-bdfd-d23b41360b0e (for exercises 1 & bonus)

This notebook uses **IfcOpenShell**, a Python library for reading and writing IFC files.

## Setup: Load and Explore IFC Files

First, let's install IfcOpenShell and load the duplex apartment model.

In [2]:
# Install IfcOpenShell (run this once)
import subprocess
import sys

try:
    import ifcopenshell
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "ifcopenshell", "-q"])
    import ifcopenshell

# Import other useful libraries
import json
from pathlib import Path
from collections import defaultdict

# Load the IFC file
ifc_file_path = Path("assets/duplex.ifc")
ifc = ifcopenshell.open(ifc_file_path)

print(f"✓ IFC file loaded: {ifc_file_path}")
print(f"✓ IFC Schema: {ifc.schema}")

✓ IFC file loaded: assets\duplex.ifc
✓ IFC Schema: IFC2X3


### What's Inside the File?

Let's explore the structure of the IFC model:

In [3]:
# Get all unique entity types in the model
all_entities = list(ifc)
entity_types = defaultdict(int)
for entity in all_entities:
    entity_types[entity.is_a()] += 1

# Show entity counts
print("Entity types in this model:")
print("-" * 40)
for entity_type in sorted(entity_types.keys()):
    count = entity_types[entity_type]
    print(f"  {entity_type}: {count}")

print(f"\nTotal entities: {len(all_entities)}")

# Find basic info
building = ifc.by_type("IfcBuilding")[0] if ifc.by_type("IfcBuilding") else None
if building:
    print(f"\nBuilding name: {building.Name}")
    print(f"Building description: {building.Description}")

Entity types in this model:
----------------------------------------
  IfcApplication: 1
  IfcArbitraryClosedProfileDef: 102
  IfcArbitraryOpenProfileDef: 184
  IfcArbitraryProfileDefWithVoids: 18
  IfcAxis2Placement2D: 397
  IfcAxis2Placement3D: 1279
  IfcBeam: 8
  IfcBooleanClippingResult: 7
  IfcBuilding: 1
  IfcBuildingStorey: 4
  IfcCartesianPoint: 8520
  IfcCartesianTransformationOperator3D: 167
  IfcCircle: 96
  IfcCircleProfileDef: 8
  IfcColourRgb: 43
  IfcCompositeCurve: 44
  IfcCompositeCurveSegment: 322
  IfcConnectedFaceSet: 235
  IfcConnectionSurfaceGeometry: 265
  IfcConversionBasedUnit: 1
  IfcCovering: 13
  IfcCurveBoundedPlane: 81
  IfcCurveStyle: 31
  IfcCurveStyleFont: 19
  IfcCurveStyleFontPattern: 57
  IfcDimensionalExponents: 1
  IfcDirection: 134
  IfcDoor: 14
  IfcDoorLiningProperties: 6
  IfcDoorStyle: 6
  IfcDraughtingPreDefinedCurveFont: 1
  IfcElementQuantity: 21
  IfcExtrudedAreaSolid: 421
  IfcFace: 4486
  IfcFaceBasedSurfaceModel: 40
  IfcFaceBound: 60
 

## Example: Extract and Display All IFC Spaces

An **IfcSpace** represents a room or area in the building. Each space has properties like name, area, and volume. Here's how to extract them:

In [4]:
# Get all spaces
spaces = ifc.by_type("IfcSpace")

print(f"Found {len(spaces)} spaces in the building\n")
print("=" * 60)

for i, space in enumerate(spaces, 1):
    # Extract properties
    name = space.Name if space.Name else "Unnamed"
    
    # Try to get area and volume
    area = None
    volume = None
    
    if hasattr(space, 'Quantities') and space.Quantities:
        for quantity in space.Quantities.Quantities:
            if hasattr(quantity, 'Name'):
                if quantity.Name == "NetFloorArea" and hasattr(quantity, 'AreaValue'):
                    area = quantity.AreaValue
                elif quantity.Name == "GrossVolume" and hasattr(quantity, 'VolumeValue'):
                    volume = quantity.VolumeValue
    
    # Format output
    print(f"{i}. {name}")
    if area:
        print(f"   Area: {area:.2f} m²")
    if volume:
        print(f"   Volume: {volume:.2f} m³")
    print()

print("=" * 60)

Found 21 spaces in the building

1. A101

2. A201

3. B104

4. B101

5. B201

6. A205

7. B205

8. A105

9. B105

10. R301

11. B102

12. A102

13. A103

14. B103

15. B204

16. A204

17. A104

18. B203

19. A202

20. B202

21. A203



## Exercise 1: Building Code Compliance Checker (Catalonia)

**Objective:** Write a function that validates spaces against Catalonia building code requirements.

**Reference:** [Catalan Building Code](https://notebooklm.google.com/notebook/216b245f-0fc1-4063-bdfd-d23b41360b0e)

### Key Requirements to Check

Based on the Catalan building code, residential spaces should meet these minimum standards:

| Space Type | Min Height | Min Area | Purpose |
|---|---|---|---|
| Living Room | 2.6 m | 16 m² | Main living area |
| Bedroom | 2.6 m | 9 m² | Single occupancy |
| Kitchen | 2.6 m | 8 m² | Cooking/dining |
| Bathroom | 2.3 m | 4 m² | Hygiene |
| Corridor | 2.3 m | 1.5 m² | Circulation |

### Your Task

Write a function `check_space_compliance(spaces)` that:

1. **Identifies** each space type (you can infer from the name or classify them)
2. **Extracts** the height and area properties from each space
3. **Compares** against the requirements above
4. **Reports** which spaces pass/fail and why
5. **Returns** a summary with compliance status

**Hints:**
- Height can be extracted from the Z-coordinate range of the space geometry
- Area is usually stored in space.Quantities as "NetFloorArea"
- Space names often indicate their type (e.g., "Master Bedroom", "Kitchen")

### Starter Code

In [18]:
def check_space_compliance(spaces):
    """
    Check each space for Catalan building code compliance.
    """
    requirements = {
        "Living Room": {"min_height": 2.6, "min_area": 16},
        "Bedroom": {"min_height": 2.6, "min_area": 9},
        "Kitchen": {"min_height": 2.6, "min_area": 8},
        "Bathroom": {"min_height": 2.3, "min_area": 4},
        "Corridor": {"min_height": 2.3, "min_area": 1.5},
    }

    report = {"passed": [], "failed": [], "warnings": []}

    def to_float(v):
        try:
            return float(v)
        except Exception:
            return None

    def classify_space_type(space):
        parts = [
            getattr(space, "Name", "") or "",
            getattr(space, "LongName", "") or "",
            getattr(space, "ObjectType", "") or "",
            str(getattr(space, "PredefinedType", "") or ""),
        ]
        text = " ".join(parts).lower()

        mapping = {
            "Living Room": ["living", "lounge", "sala", "salon", "estar", "menjador"],
            "Bedroom": ["bedroom", "bed", "dorm", "habitacio", "habitació", "dormitori"],
            "Kitchen": ["kitchen", "cuina", "cocina"],
            "Bathroom": ["bath", "bathroom", "toilet", "wc", "lavabo", "aseo", "bany"],
            "Corridor": ["corridor", "hall", "pasillo", "passage", "rebedor", "distrib"],
        }

        for space_type, keywords in mapping.items():
            if any(k in text for k in keywords):
                return space_type
        return None

    def extract_area(space):
        for rel in getattr(space, "IsDefinedBy", []) or []:
            pdef = getattr(rel, "RelatingPropertyDefinition", None)
            if pdef and pdef.is_a("IfcElementQuantity"):
                for q in getattr(pdef, "Quantities", []) or []:
                    qn = (getattr(q, "Name", "") or "").lower()
                    if q.is_a("IfcQuantityArea") and qn in ["netfloorarea", "grossfloorarea", "floorarea"]:
                        return to_float(getattr(q, "AreaValue", None))

        for rel in getattr(space, "IsDefinedBy", []) or []:
            pdef = getattr(rel, "RelatingPropertyDefinition", None)
            if pdef and pdef.is_a("IfcPropertySet"):
                for p in getattr(pdef, "HasProperties", []) or []:
                    pn = (getattr(p, "Name", "") or "").lower()
                    if "area" in pn:
                        nominal = getattr(p, "NominalValue", None)
                        return to_float(getattr(nominal, "wrappedValue", None))
        return None

    def extract_height(space):
        for rel in getattr(space, "IsDefinedBy", []) or []:
            pdef = getattr(rel, "RelatingPropertyDefinition", None)
            if pdef and pdef.is_a("IfcElementQuantity"):
                for q in getattr(pdef, "Quantities", []) or []:
                    qn = (getattr(q, "Name", "") or "").lower()
                    if q.is_a("IfcQuantityLength") and qn in ["height", "clearheight", "netheight"]:
                        return to_float(getattr(q, "LengthValue", None))

        for rel in getattr(space, "IsDefinedBy", []) or []:
            pdef = getattr(rel, "RelatingPropertyDefinition", None)
            if pdef and pdef.is_a("IfcPropertySet"):
                for p in getattr(pdef, "HasProperties", []) or []:
                    pn = (getattr(p, "Name", "") or "").lower()
                    if "height" in pn:
                        nominal = getattr(p, "NominalValue", None)
                        return to_float(getattr(nominal, "wrappedValue", None))
        return None

    for space in spaces:
        name = getattr(space, "Name", None) or "Unnamed Space"
        space_type = classify_space_type(space)
        area = extract_area(space)
        height = extract_height(space)
        reasons = []

        if space_type is None:
            reasons.append("Could not infer space type from Name/LongName/ObjectType.")
            report["failed"].append({
                "space": name,
                "space_type": "Unknown",
                "measured": {"area_m2": area, "height_m": height},
                "required": {},
                "status": "FAIL",
                "reasons": reasons
            })
            continue

        req = requirements[space_type]

        if area is None:
            reasons.append("Area not found.")
        elif area < req["min_area"]:
            reasons.append(f"Area {area:.2f} m2 < required {req['min_area']:.2f} m2.")

        if height is None:
            reasons.append("Height not found.")
        elif height < req["min_height"]:
            reasons.append(f"Height {height:.2f} m < required {req['min_height']:.2f} m.")

        item = {
            "space": name,
            "space_type": space_type,
            "measured": {"area_m2": area, "height_m": height},
            "required": {"min_area_m2": req["min_area"], "min_height_m": req["min_height"]},
            "status": "FAIL" if reasons else "PASS",
            "reasons": reasons if reasons else ["Meets minimum area and height requirements."]
        }

        if reasons:
            report["failed"].append(item)
        else:
            report["passed"].append(item)

    total_checked = len(report["passed"]) + len(report["failed"])
    report["summary"] = {
        "total_spaces_input": len(spaces),
        "total_spaces_checked": total_checked,
        "passed_count": len(report["passed"]),
        "failed_count": len(report["failed"]),
        "warnings_count": len(report["warnings"]),
        "compliance_rate_percent": round((len(report["passed"]) / total_checked * 100), 2) if total_checked else 0.0
    }

    return report


def build_full_report_text(result):
    s = result["summary"]
    passed = result["passed"]
    failed = result["failed"]
    warnings = result["warnings"]

    lines = []
    lines.append("=" * 96)
    lines.append("CATALAN BUILDING CODE - SPACE COMPLIANCE REPORT")
    lines.append("=" * 96)
    lines.append(f"Total spaces input   : {s['total_spaces_input']}")
    lines.append(f"Total spaces checked : {s['total_spaces_checked']}")
    lines.append(f"Passed               : {s['passed_count']}")
    lines.append(f"Failed               : {s['failed_count']}")
    lines.append(f"Warnings             : {s['warnings_count']}")
    lines.append(f"Compliance rate      : {s['compliance_rate_percent']}%")
    lines.append("-" * 96)

    # Passed spaces
    lines.append("")
    lines.append("PASSED SPACES (NAMES + DETAILS)")
    lines.append("-" * 96)
    if not passed:
        lines.append("None")
    else:
        header = f"{'Space':25} {'Type':12} {'Area(m2)':>10} {'Height(m)':>10} {'ReqArea':>10} {'ReqH':>8} {'Status':>8}"
        lines.append(header)
        lines.append("-" * len(header))
        for p in passed:
            m = p.get("measured", {})
            r = p.get("required", {})
            area = f"{m.get('area_m2'):.2f}" if isinstance(m.get("area_m2"), (int, float)) else "N/A"
            hgt = f"{m.get('height_m'):.2f}" if isinstance(m.get("height_m"), (int, float)) else "N/A"
            req_a = f"{r.get('min_area_m2'):.2f}" if isinstance(r.get("min_area_m2"), (int, float)) else "N/A"
            req_h = f"{r.get('min_height_m'):.2f}" if isinstance(r.get("min_height_m"), (int, float)) else "N/A"
            lines.append(
                f"{p.get('space','N/A')[:25]:25} {p.get('space_type','N/A')[:12]:12} "
                f"{area:>10} {hgt:>10} {req_a:>10} {req_h:>8} {p.get('status','N/A'):>8}"
            )

    # Failed spaces
    lines.append("")
    lines.append("FAILED SPACES (NAMES + DETAILS + WHY FAILED)")
    lines.append("-" * 96)
    if not failed:
        lines.append("None")
    else:
        header = f"{'Space':25} {'Type':12} {'Area(m2)':>10} {'Height(m)':>10} {'ReqArea':>10} {'ReqH':>8} {'Status':>8}"
        lines.append(header)
        lines.append("-" * len(header))
        for f in failed:
            m = f.get("measured", {})
            r = f.get("required", {})
            area = f"{m.get('area_m2'):.2f}" if isinstance(m.get("area_m2"), (int, float)) else "N/A"
            hgt = f"{m.get('height_m'):.2f}" if isinstance(m.get("height_m"), (int, float)) else "N/A"
            req_a = f"{r.get('min_area_m2'):.2f}" if isinstance(r.get("min_area_m2"), (int, float)) else "N/A"
            req_h = f"{r.get('min_height_m'):.2f}" if isinstance(r.get("min_height_m"), (int, float)) else "N/A"
            lines.append(
                f"{f.get('space','N/A')[:25]:25} {f.get('space_type','N/A')[:12]:12} "
                f"{area:>10} {hgt:>10} {req_a:>10} {req_h:>8} {f.get('status','N/A'):>8}"
            )

        lines.append("")
        lines.append("FAILURE REASONS")
        lines.append("-" * 96)
        for i, f in enumerate(failed, 1):
            lines.append(f"{i}. {f.get('space', 'N/A')} ({f.get('space_type', 'Unknown')})")
            for reason in f.get("reasons", []):
                lines.append(f"   - {reason}")

    if warnings:
        lines.append("")
        lines.append("WARNINGS")
        lines.append("-" * 96)
        for i, w in enumerate(warnings, 1):
            lines.append(f"{i}. {w.get('space','N/A')}: {w.get('reason','No reason provided')}")

    lines.append("=" * 96)
    return "\n".join(lines)


# Run
result = check_space_compliance(spaces)
report_text = build_full_report_text(result)

# 1) Print to notebook
print(report_text)

# 2) Save full report to file (no truncation)
with open("compliance_report.txt", "w", encoding="utf-8") as f:
    f.write(report_text)

print("\nSaved full report to: compliance_report.txt")


CATALAN BUILDING CODE - SPACE COMPLIANCE REPORT
Total spaces input   : 21
Total spaces checked : 21
Passed               : 12
Failed               : 9
Warnings             : 0
Compliance rate      : 57.14%
------------------------------------------------------------------------------------------------

PASSED SPACES (NAMES + DETAILS)
------------------------------------------------------------------------------------------------
Space                     Type           Area(m2)  Height(m)    ReqArea     ReqH   Status
-----------------------------------------------------------------------------------------
A201                      Corridor           7.80       2.90       1.50     2.30     PASS
B201                      Corridor           7.80       2.90       1.50     2.30     PASS
B102                      Living Room       30.14       2.60      16.00     2.60     PASS
A102                      Living Room       30.14       2.60      16.00     2.60     PASS
A103                      K

## Exercise 2: Window Detection and Compliance Verification

**Objective:** Find all windows in the model and verify they meet natural light and ventilation requirements.

**Reference:** [Catalan Building Code](https://notebooklm.google.com/notebook/216b245f-0fc1-4063-bdfd-d23b41360b0e)

### Key Requirements

Residential spaces must have:
- **Minimum window area** = 1/8 of floor area (12.5%)
- **Minimum window dimensions** = 60cm width, 100cm height (for single opening)
- Living areas should have direct natural light

### Your Task

Write a function `analyze_window_compliance(ifc_model, spaces)` that:

1. **Finds all IfcWindow** entities in the model
2. **Links windows to spaces** (which room is each window in?)
3. **Extracts properties**: dimensions, area, orientation
4. **Calculates window-to-floor ratio** for each space
5. **Reports compliance** for each space with windows

**Hints:**
- Windows are `IfcWindow` entities
- To link windows to spaces, check spatial containment relationships
- Window area can be calculated from the frame/pane dimensions
- You may need to examine the geometric representation

### Starter Code

In [20]:
# Exercise 2: Complete code
# - analyzes window compliance
# - prints structured output
# - appends full window report to existing compliance_report.txt (without deleting old content)

def analyze_window_compliance(ifc_model, spaces):
    """
    Analyze windows in each space and check compliance.

    Args:
        ifc_model: The loaded IFC model
        spaces: List of IfcSpace objects

    Returns:
        dict: Report with window analysis and compliance status
    """
    windows = ifc_model.by_type("IfcWindow")

    report = {
        "total_windows": len(windows),
        "windows_by_space": {},
        "compliance_status": {},
        "unassigned_windows": [],
        "compliance": {}
    }

    def to_float(v):
        try:
            return float(v)
        except Exception:
            return None

    def classify_space_type(space):
        parts = [
            getattr(space, "Name", "") or "",
            getattr(space, "LongName", "") or "",
            getattr(space, "ObjectType", "") or "",
            str(getattr(space, "PredefinedType", "") or ""),
        ]
        text = " ".join(parts).lower()
        if any(k in text for k in ["living", "lounge", "sala", "salon", "estar", "menjador"]):
            return "Living Room"
        if any(k in text for k in ["bedroom", "bed", "dorm", "habitacio", "habitació", "dormitori"]):
            return "Bedroom"
        if any(k in text for k in ["kitchen", "cuina", "cocina"]):
            return "Kitchen"
        if any(k in text for k in ["bath", "bathroom", "toilet", "wc", "lavabo", "aseo", "bany"]):
            return "Bathroom"
        if any(k in text for k in ["corridor", "hall", "pasillo", "passage", "rebedor", "distrib"]):
            return "Corridor"
        return "Unknown"

    def extract_space_area(space):
        # IfcElementQuantity first
        for rel in getattr(space, "IsDefinedBy", []) or []:
            pdef = getattr(rel, "RelatingPropertyDefinition", None)
            if pdef and pdef.is_a("IfcElementQuantity"):
                for q in getattr(pdef, "Quantities", []) or []:
                    qn = (getattr(q, "Name", "") or "").lower()
                    if q.is_a("IfcQuantityArea") and qn in ["netfloorarea", "grossfloorarea", "floorarea"]:
                        return to_float(getattr(q, "AreaValue", None))
        # Property fallback
        for rel in getattr(space, "IsDefinedBy", []) or []:
            pdef = getattr(rel, "RelatingPropertyDefinition", None)
            if pdef and pdef.is_a("IfcPropertySet"):
                for p in getattr(pdef, "HasProperties", []) or []:
                    pn = (getattr(p, "Name", "") or "").lower()
                    if "area" in pn:
                        nominal = getattr(p, "NominalValue", None)
                        return to_float(getattr(nominal, "wrappedValue", None))
        return None

    def extract_window_dimensions(window):
        w = to_float(getattr(window, "OverallWidth", None))
        h = to_float(getattr(window, "OverallHeight", None))
        return w, h

    def extract_window_area(window, width, height):
        if width is not None and height is not None:
            return width * height
        # Quantity fallback
        for rel in getattr(window, "IsDefinedBy", []) or []:
            pdef = getattr(rel, "RelatingPropertyDefinition", None)
            if pdef and pdef.is_a("IfcElementQuantity"):
                for q in getattr(pdef, "Quantities", []) or []:
                    if q.is_a("IfcQuantityArea"):
                        return to_float(getattr(q, "AreaValue", None))
        return None

    def extract_window_orientation(window):
        # Best-effort orientation from local placement direction
        try:
            placement = getattr(window, "ObjectPlacement", None)
            rel_placement = getattr(placement, "RelativePlacement", None)
            ref_dir = getattr(rel_placement, "RefDirection", None)
            ratios = getattr(ref_dir, "DirectionRatios", None)
            if ratios and len(ratios) >= 2:
                x = to_float(ratios[0])
                y = to_float(ratios[1])
                if x is not None and y is not None:
                    import math
                    deg = (math.degrees(math.atan2(y, x)) + 360) % 360
                    return f"{deg:.1f}°"
        except Exception:
            pass
        return "Unknown"

    # Build space map
    for s in spaces:
        sgid = getattr(s, "GlobalId", None)
        if not sgid:
            continue
        report["windows_by_space"][sgid] = {
            "space_name": getattr(s, "Name", None) or "Unnamed Space",
            "space_type": classify_space_type(s),
            "floor_area_m2": extract_space_area(s),
            "windows": [],
            "total_window_area_m2": 0.0,
            "window_to_floor_ratio": None
        }

    # Link windows to spaces
    window_to_spaces = {}  # window global id -> set of space global ids

    def add_link(window, space_obj):
        wgid = getattr(window, "GlobalId", None)
        sgid = getattr(space_obj, "GlobalId", None)
        if wgid and sgid and sgid in report["windows_by_space"]:
            window_to_spaces.setdefault(wgid, set()).add(sgid)

    # Method A: direct containment
    for w in windows:
        for rel in getattr(w, "ContainedInStructure", []) or []:
            s = getattr(rel, "RelatingStructure", None)
            if s and s.is_a("IfcSpace"):
                add_link(w, s)

    # Method B: space boundaries
    for s in spaces:
        for b in getattr(s, "BoundedBy", []) or []:
            elem = getattr(b, "RelatedBuildingElement", None)
            if elem and elem.is_a("IfcWindow"):
                add_link(elem, s)

    # Fill window details into spaces
    for w in windows:
        wgid = getattr(w, "GlobalId", None)
        wname = getattr(w, "Name", None) or "Unnamed Window"
        width, height = extract_window_dimensions(w)
        area = extract_window_area(w, width, height)
        orientation = extract_window_orientation(w)

        wdetail = {
            "window_id": wgid,
            "window_name": wname,
            "width_m": width,
            "height_m": height,
            "area_m2": area,
            "orientation": orientation
        }

        linked_spaces = list(window_to_spaces.get(wgid, []))
        if not linked_spaces:
            report["unassigned_windows"].append(wdetail)
            continue

        for sgid in linked_spaces:
            report["windows_by_space"][sgid]["windows"].append(wdetail)
            if isinstance(area, (int, float)):
                report["windows_by_space"][sgid]["total_window_area_m2"] += area

    # Compliance checks
    compliant_count = 0
    non_compliant_count = 0
    spaces_with_windows = 0

    for sgid, sdata in report["windows_by_space"].items():
        floor_area = sdata["floor_area_m2"]
        total_window_area = sdata["total_window_area_m2"]
        windows_list = sdata["windows"]
        space_type = sdata["space_type"]

        reasons = []
        ratio = None

        if windows_list:
            spaces_with_windows += 1

        # Ratio >= 1/8
        if isinstance(floor_area, (int, float)) and floor_area > 0:
            ratio = total_window_area / floor_area
            sdata["window_to_floor_ratio"] = ratio
            if ratio < 0.125:
                reasons.append(f"Window-to-floor ratio {ratio:.3f} < 0.125 (12.5%).")
        else:
            reasons.append("Floor area missing; cannot verify window-to-floor ratio.")

        # Minimum window size per opening
        for w in windows_list:
            ww = w.get("width_m")
            hh = w.get("height_m")
            if ww is None or hh is None:
                reasons.append(f"{w['window_name']}: missing width/height.")
            elif ww < 0.6 or hh < 1.0:
                reasons.append(f"{w['window_name']}: {ww:.2f}m x {hh:.2f}m < 0.60m x 1.00m.")

        # Habitable spaces should have windows
        if space_type in ["Living Room", "Bedroom", "Kitchen"] and len(windows_list) == 0:
            reasons.append("Habitable room has no window (no direct natural light).")

        status = "PASS" if len(reasons) == 0 else "FAIL"
        if status == "PASS":
            compliant_count += 1
        else:
            non_compliant_count += 1

        report["compliance_status"][sgid] = {
            "space_name": sdata["space_name"],
            "space_type": space_type,
            "status": status,
            "window_count": len(windows_list),
            "total_window_area_m2": total_window_area,
            "floor_area_m2": floor_area,
            "window_to_floor_ratio": ratio,
            "reasons": reasons
        }

    report["compliance"] = {
        "total_spaces_analyzed": len(report["windows_by_space"]),
        "spaces_with_windows": spaces_with_windows,
        "compliant_spaces": compliant_count,
        "non_compliant_spaces": non_compliant_count,
        "unassigned_windows_count": len(report["unassigned_windows"])
    }

    return report


def build_window_report_text(analysis):
    lines = []
    lines.append("\n" + "=" * 96)
    lines.append("WINDOW COMPLIANCE REPORT")
    lines.append("=" * 96)
    lines.append(f"Total windows in model        : {analysis['total_windows']}")
    lines.append(f"Total spaces analyzed         : {analysis['compliance']['total_spaces_analyzed']}")
    lines.append(f"Spaces with windows           : {analysis['compliance']['spaces_with_windows']}")
    lines.append(f"Compliant spaces              : {analysis['compliance']['compliant_spaces']}")
    lines.append(f"Non-compliant spaces          : {analysis['compliance']['non_compliant_spaces']}")
    lines.append(f"Unassigned windows            : {analysis['compliance']['unassigned_windows_count']}")
    lines.append("-" * 96)

    passed = []
    failed = []
    for _, data in analysis["compliance_status"].items():
        if data["status"] == "PASS":
            passed.append(data)
        else:
            failed.append(data)

    def fmt(v, nd=3):
        if isinstance(v, (int, float)):
            return f"{v:.{nd}f}"
        return "N/A"

    lines.append("\nPASSED SPACES")
    lines.append("-" * 96)
    if not passed:
        lines.append("None")
    else:
        header = f"{'Space':25} {'Type':12} {'Win#':>5} {'Win.Area(m2)':>12} {'Floor(m2)':>10} {'Ratio':>8}"
        lines.append(header)
        lines.append("-" * len(header))
        for p in passed:
            lines.append(
                f"{p['space_name'][:25]:25} {p['space_type'][:12]:12} "
                f"{p['window_count']:>5} {fmt(p['total_window_area_m2'],2):>12} "
                f"{fmt(p['floor_area_m2'],2):>10} {fmt(p['window_to_floor_ratio'],3):>8}"
            )

    lines.append("\nFAILED SPACES (WITH REASONS)")
    lines.append("-" * 96)
    if not failed:
        lines.append("None")
    else:
        header = f"{'Space':25} {'Type':12} {'Win#':>5} {'Win.Area(m2)':>12} {'Floor(m2)':>10} {'Ratio':>8}"
        lines.append(header)
        lines.append("-" * len(header))
        for f in failed:
            lines.append(
                f"{f['space_name'][:25]:25} {f['space_type'][:12]:12} "
                f"{f['window_count']:>5} {fmt(f['total_window_area_m2'],2):>12} "
                f"{fmt(f['floor_area_m2'],2):>10} {fmt(f['window_to_floor_ratio'],3):>8}"
            )
        lines.append("\nFailure Details:")
        for i, f in enumerate(failed, 1):
            lines.append(f"{i}. {f['space_name']} ({f['space_type']})")
            for r in f["reasons"]:
                lines.append(f"   - {r}")

    if analysis["unassigned_windows"]:
        lines.append("\nUNASSIGNED WINDOWS")
        lines.append("-" * 96)
        for i, w in enumerate(analysis["unassigned_windows"], 1):
            lines.append(
                f"{i}. {w['window_name']} | "
                f"size={w.get('width_m','N/A')} x {w.get('height_m','N/A')} m | "
                f"area={w.get('area_m2','N/A')} m2 | orientation={w.get('orientation','Unknown')}"
            )

    lines.append("=" * 96)
    return "\n".join(lines)


def print_window_compliance_report(analysis):
    print(build_window_report_text(analysis))


# Run analysis
analysis = analyze_window_compliance(ifc, spaces)

# Print in notebook
print_window_compliance_report(analysis)

# Append to existing compliance_report.txt (does NOT overwrite old Exercise 1 content)
window_report_text = build_window_report_text(analysis)
with open("compliance_report.txt", "a", encoding="utf-8") as f:
    f.write("\n\n")
    f.write(window_report_text)

print("\nWindow compliance results appended to compliance_report.txt")



WINDOW COMPLIANCE REPORT
Total windows in model        : 24
Total spaces analyzed         : 21
Spaces with windows           : 8
Compliant spaces              : 2
Non-compliant spaces          : 19
Unassigned windows            : 10
------------------------------------------------------------------------------------------------

PASSED SPACES
------------------------------------------------------------------------------------------------
Space                     Type          Win# Win.Area(m2)  Floor(m2)    Ratio
-----------------------------------------------------------------------------
B102                      Living Room      2        13.35      30.14    0.443
A102                      Living Room      2        13.35      30.14    0.443

FAILED SPACES (WITH REASONS)
------------------------------------------------------------------------------------------------
Space                     Type          Win# Win.Area(m2)  Floor(m2)    Ratio
----------------------------------------

## Bonus Exercise: Fire Safety Route Analysis

**Objective:** Find the longest evacuation route within the apartment and verify it meets fire safety requirements.

**Difficulty:** Advanced

**Reference:** [Catalan Building Code - Fire Safety Section](https://notebooklm.google.com/notebook/216b245f-0fc1-4063-bdfd-d23b41360b0e)

### Fire Safety Requirements

According to the Catalan building code:
- **Maximum travel distance** to exit: ≤ 25-30 m (depending on building type)
- **Minimum corridor width**: 1.2 m
- **Minimum door width**: 0.8 m (for exits)
- **No dead-end corridors** longer than 10 m

### Your Task

Write a function `analyze_evacuation_routes(ifc_model, spaces)` that:

1. **Builds a spatial graph** of the apartment (rooms as nodes, doors/openings as connections)
2. **Calculates distances** between spaces (using area/perimeter as proxy)
3. **Finds the longest route** from any point to the nearest exit
4. **Validates** the route against fire safety requirements
5. **Identifies bottlenecks** (narrow corridors, small doors)

**Hints:**
- Think of spaces as nodes and doors as edges in a graph
- Use BFS/DFS to find longest paths
- Door dimensions can indicate width constraints
- Consider calculating distances based on space geometry
- This is a simplified model - real analysis would use detailed geometry

### Starter Code

```python
def analyze_evacuation_routes(ifc_model, spaces):
    """
    Analyze evacuation routes and fire safety compliance.
    
    Args:
        ifc_model: The loaded IFC model
        spaces: List of IfcSpace objects
        
    Returns:
        dict: Fire safety analysis and recommendations
    """
    
    # Get all doors (potential connections between spaces)
    doors = ifc_model.by_type("IfcDoor")
    
    analysis = {
        "total_spaces": len(spaces),
        "total_doors": len(doors),
        "longest_route": None,
        "longest_distance": 0,
        "safety_issues": [],
        "compliant": False
    }
    
    print(f"Analyzing {len(spaces)} spaces with {len(doors)} doors")
    
    # TODO: Build spatial connectivity graph
    # TODO: Find longest evacuation path
    # TODO: Check corridor widths and door dimensions
    # TODO: Validate against requirements
    # TODO: Report issues and recommendations
    
    return analysis


# Run the analysis (uncomment when ready)
# fire_analysis = analyze_evacuation_routes(ifc, spaces)
```

In [21]:
# Bonus Exercise: Fire Safety Route Analysis (complete code)

from collections import defaultdict
import heapq
import math

def analyze_evacuation_routes(ifc_model, spaces):
    """
    Analyze evacuation routes and fire safety compliance.

    Args:
        ifc_model: The loaded IFC model
        spaces: List of IfcSpace objects

    Returns:
        dict: Fire safety analysis and recommendations
    """
    doors = ifc_model.by_type("IfcDoor")

    analysis = {
        "total_spaces": len(spaces),
        "total_doors": len(doors),
        "longest_route": None,
        "longest_distance": 0.0,
        "exit_spaces": [],
        "graph_nodes": 0,
        "graph_edges": 0,
        "door_issues": [],
        "corridor_issues": [],
        "dead_end_issues": [],
        "safety_issues": [],
        "compliant": False
    }

    def to_float(v):
        try:
            return float(v)
        except Exception:
            return None

    def space_name(space):
        return getattr(space, "Name", None) or "Unnamed Space"

    def classify_space_type(space):
        parts = [
            getattr(space, "Name", "") or "",
            getattr(space, "LongName", "") or "",
            getattr(space, "ObjectType", "") or "",
            str(getattr(space, "PredefinedType", "") or ""),
        ]
        text = " ".join(parts).lower()
        if any(k in text for k in ["corridor", "hall", "pasillo", "passage", "rebedor", "distrib"]):
            return "Corridor"
        if any(k in text for k in ["living", "lounge", "sala", "salon", "estar", "menjador"]):
            return "Living Room"
        if any(k in text for k in ["bedroom", "bed", "dorm", "habitacio", "habitació", "dormitori"]):
            return "Bedroom"
        if any(k in text for k in ["kitchen", "cuina", "cocina"]):
            return "Kitchen"
        if any(k in text for k in ["bath", "bathroom", "toilet", "wc", "lavabo", "aseo", "bany"]):
            return "Bathroom"
        return "Unknown"

    def extract_space_area(space):
        # IfcElementQuantity first
        for rel in getattr(space, "IsDefinedBy", []) or []:
            pdef = getattr(rel, "RelatingPropertyDefinition", None)
            if pdef and pdef.is_a("IfcElementQuantity"):
                for q in getattr(pdef, "Quantities", []) or []:
                    qn = (getattr(q, "Name", "") or "").lower()
                    if q.is_a("IfcQuantityArea") and qn in ["netfloorarea", "grossfloorarea", "floorarea"]:
                        return to_float(getattr(q, "AreaValue", None))
        # Property fallback
        for rel in getattr(space, "IsDefinedBy", []) or []:
            pdef = getattr(rel, "RelatingPropertyDefinition", None)
            if pdef and pdef.is_a("IfcPropertySet"):
                for p in getattr(pdef, "HasProperties", []) or []:
                    pn = (getattr(p, "Name", "") or "").lower()
                    if "area" in pn:
                        nominal = getattr(p, "NominalValue", None)
                        return to_float(getattr(nominal, "wrappedValue", None))
        return None

    def extract_space_width(space):
        # width from quantity/property if available
        for rel in getattr(space, "IsDefinedBy", []) or []:
            pdef = getattr(rel, "RelatingPropertyDefinition", None)
            if pdef and pdef.is_a("IfcElementQuantity"):
                for q in getattr(pdef, "Quantities", []) or []:
                    qn = (getattr(q, "Name", "") or "").lower()
                    if q.is_a("IfcQuantityLength") and qn in ["width", "clearwidth", "netwidth"]:
                        return to_float(getattr(q, "LengthValue", None))
        for rel in getattr(space, "IsDefinedBy", []) or []:
            pdef = getattr(rel, "RelatingPropertyDefinition", None)
            if pdef and pdef.is_a("IfcPropertySet"):
                for p in getattr(pdef, "HasProperties", []) or []:
                    pn = (getattr(p, "Name", "") or "").lower()
                    if "width" in pn:
                        nominal = getattr(p, "NominalValue", None)
                        return to_float(getattr(nominal, "wrappedValue", None))
        return None

    def extract_door_width(door):
        # primary
        w = to_float(getattr(door, "OverallWidth", None))
        if w is not None:
            return w
        # fallback from quantities/properties
        for rel in getattr(door, "IsDefinedBy", []) or []:
            pdef = getattr(rel, "RelatingPropertyDefinition", None)
            if pdef and pdef.is_a("IfcElementQuantity"):
                for q in getattr(pdef, "Quantities", []) or []:
                    qn = (getattr(q, "Name", "") or "").lower()
                    if q.is_a("IfcQuantityLength") and qn in ["width", "clearwidth", "netwidth"]:
                        return to_float(getattr(q, "LengthValue", None))
        for rel in getattr(door, "IsDefinedBy", []) or []:
            pdef = getattr(rel, "RelatingPropertyDefinition", None)
            if pdef and pdef.is_a("IfcPropertySet"):
                for p in getattr(pdef, "HasProperties", []) or []:
                    pn = (getattr(p, "Name", "") or "").lower()
                    if "width" in pn:
                        nominal = getattr(p, "NominalValue", None)
                        return to_float(getattr(nominal, "wrappedValue", None))
        return None

    def is_exit_door(door):
        nm = (getattr(door, "Name", "") or "").lower()
        if any(k in nm for k in ["exit", "entrance", "entry", "main", "front", "outside", "external"]):
            return True
        return False

    # Space registry
    spaces_by_gid = {}
    space_meta = {}
    for s in spaces:
        gid = getattr(s, "GlobalId", None)
        if not gid:
            continue
        area = extract_space_area(s)
        stype = classify_space_type(s)
        space_meta[gid] = {
            "name": space_name(s),
            "type": stype,
            "area_m2": area,
            "width_m": extract_space_width(s)
        }
        spaces_by_gid[gid] = s

    # Build door -> spaces map from space boundaries
    door_to_spaces = defaultdict(set)
    doors_by_gid = {}
    for d in doors:
        dgid = getattr(d, "GlobalId", None)
        if dgid:
            doors_by_gid[dgid] = d

    for s in spaces:
        sgid = getattr(s, "GlobalId", None)
        if not sgid:
            continue
        for b in getattr(s, "BoundedBy", []) or []:
            elem = getattr(b, "RelatedBuildingElement", None)
            if elem and elem.is_a("IfcDoor"):
                dgid = getattr(elem, "GlobalId", None)
                if dgid:
                    door_to_spaces[dgid].add(sgid)

    # Graph
    graph = defaultdict(list)  # sgid -> list[(neighbor_gid, dist, door_gid)]

    def proxy_edge_distance(sgid_a, sgid_b):
        # simplified distance proxy based on room size
        a1 = space_meta.get(sgid_a, {}).get("area_m2")
        a2 = space_meta.get(sgid_b, {}).get("area_m2")
        if isinstance(a1, (int, float)) and a1 > 0 and isinstance(a2, (int, float)) and a2 > 0:
            return max(1.0, (math.sqrt(a1) + math.sqrt(a2)) / 2.0)
        return 3.0  # fallback default

    # Identify exits and build edges
    exit_spaces = set()
    for dgid, linked_spaces in door_to_spaces.items():
        linked_spaces = list(linked_spaces)
        door = doors_by_gid.get(dgid)
        dname = getattr(door, "Name", None) or "Unnamed Door"
        dwidth = extract_door_width(door) if door else None

        # door width checks
        if dwidth is None:
            analysis["door_issues"].append(f"{dname}: width missing (cannot verify >= 0.8 m).")
        elif dwidth < 0.8:
            analysis["door_issues"].append(f"{dname}: width {dwidth:.2f} m < 0.80 m.")

        # exit candidate rules:
        # 1) door explicitly marked by name
        # 2) door linked to only one space (likely external boundary)
        explicit_exit = is_exit_door(door) if door else False
        if len(linked_spaces) == 1:
            exit_spaces.add(linked_spaces[0])
        if explicit_exit:
            for sg in linked_spaces:
                exit_spaces.add(sg)

        # connections between spaces via shared door
        if len(linked_spaces) >= 2:
            for i in range(len(linked_spaces)):
                for j in range(i + 1, len(linked_spaces)):
                    a = linked_spaces[i]
                    b = linked_spaces[j]
                    dist = proxy_edge_distance(a, b)
                    graph[a].append((b, dist, dgid))
                    graph[b].append((a, dist, dgid))

    # Corridor checks (min width 1.2m)
    for sgid, meta in space_meta.items():
        if meta["type"] == "Corridor":
            w = meta["width_m"]
            if w is None:
                analysis["corridor_issues"].append(f"{meta['name']}: corridor width missing (cannot verify >= 1.2 m).")
            elif w < 1.2:
                analysis["corridor_issues"].append(f"{meta['name']}: corridor width {w:.2f} m < 1.20 m.")

    # Dead-end corridor check (proxy): degree == 1 and estimated length > 10 m
    for sgid, meta in space_meta.items():
        if meta["type"] != "Corridor":
            continue
        degree = len(graph.get(sgid, []))
        if degree == 1:
            area = meta["area_m2"]
            width = meta["width_m"]
            est_length = None
            if isinstance(area, (int, float)) and area > 0:
                if isinstance(width, (int, float)) and width > 0:
                    est_length = area / width
                else:
                    est_length = math.sqrt(area) * 1.5
            if isinstance(est_length, (int, float)) and est_length > 10.0:
                analysis["dead_end_issues"].append(
                    f"{meta['name']}: estimated dead-end length {est_length:.2f} m > 10.0 m."
                )

    # Dijkstra helper: shortest distance from one node to any exit space
    def shortest_to_exit(start):
        if start in exit_spaces:
            return 0.0, [start]
        pq = [(0.0, start, [start])]
        visited = {}
        while pq:
            dist, node, path = heapq.heappop(pq)
            if node in visited and visited[node] <= dist:
                continue
            visited[node] = dist
            if node in exit_spaces:
                return dist, path
            for nbr, w, _dgid in graph.get(node, []):
                nd = dist + w
                if nbr not in visited or nd < visited[nbr]:
                    heapq.heappush(pq, (nd, nbr, path + [nbr]))
        return None, []

    # Longest route (max of nearest-exit distances)
    longest = -1.0
    longest_path = []
    longest_start = None

    for sgid in space_meta.keys():
        d, p = shortest_to_exit(sgid)
        if d is None:
            # disconnected from exits
            analysis["safety_issues"].append(f"{space_meta[sgid]['name']}: no route to any exit space.")
            continue
        if d > longest:
            longest = d
            longest_path = p
            longest_start = sgid

    if longest < 0:
        longest = 0.0

    analysis["longest_distance"] = float(longest)
    analysis["longest_route"] = {
        "start_space": space_meta[longest_start]["name"] if longest_start else None,
        "path_space_ids": longest_path,
        "path_space_names": [space_meta[x]["name"] for x in longest_path] if longest_path else [],
        "distance_m": float(longest)
    }
    analysis["exit_spaces"] = [space_meta[x]["name"] for x in sorted(exit_spaces) if x in space_meta]
    analysis["graph_nodes"] = len(space_meta)
    analysis["graph_edges"] = sum(len(v) for v in graph.values()) // 2

    # Max travel distance check (using conservative 25m threshold)
    if analysis["longest_distance"] > 25.0:
        analysis["safety_issues"].append(
            f"Longest travel distance {analysis['longest_distance']:.2f} m > 25.0 m."
        )

    # aggregate issues
    analysis["safety_issues"].extend(analysis["door_issues"])
    analysis["safety_issues"].extend(analysis["corridor_issues"])
    analysis["safety_issues"].extend(analysis["dead_end_issues"])

    analysis["compliant"] = len(analysis["safety_issues"]) == 0
    return analysis


def build_fire_report_text(fire_analysis):
    lines = []
    lines.append("\n" + "=" * 96)
    lines.append("FIRE SAFETY ROUTE ANALYSIS")
    lines.append("=" * 96)
    lines.append(f"Total spaces               : {fire_analysis['total_spaces']}")
    lines.append(f"Total doors                : {fire_analysis['total_doors']}")
    lines.append(f"Graph nodes                : {fire_analysis['graph_nodes']}")
    lines.append(f"Graph edges                : {fire_analysis['graph_edges']}")
    lines.append(f"Longest route distance (m) : {fire_analysis['longest_distance']:.2f}")
    lines.append(f"Compliant                  : {fire_analysis['compliant']}")
    lines.append("-" * 96)

    lines.append("Longest Route")
    lines.append("-" * 96)
    lr = fire_analysis.get("longest_route") or {}
    lines.append(f"Start space: {lr.get('start_space')}")
    names = lr.get("path_space_names", [])
    lines.append("Path: " + (" -> ".join(names) if names else "N/A"))

    lines.append("\nExit Spaces")
    lines.append("-" * 96)
    if fire_analysis["exit_spaces"]:
        for i, x in enumerate(fire_analysis["exit_spaces"], 1):
            lines.append(f"{i}. {x}")
    else:
        lines.append("No exit spaces inferred.")

    lines.append("\nSafety Issues")
    lines.append("-" * 96)
    if fire_analysis["safety_issues"]:
        for i, issue in enumerate(fire_analysis["safety_issues"], 1):
            lines.append(f"{i}. {issue}")
    else:
        lines.append("No safety issues detected.")

    lines.append("=" * 96)
    return "\n".join(lines)


def print_fire_report(fire_analysis):
    print(build_fire_report_text(fire_analysis))


# Run analysis
fire_analysis = analyze_evacuation_routes(ifc, spaces)
print_fire_report(fire_analysis)

# Append to existing compliance_report.txt without removing earlier content
fire_report_text = build_fire_report_text(fire_analysis)
with open("compliance_report.txt", "a", encoding="utf-8") as f:
    f.write("\n\n")
    f.write(fire_report_text)

print("\nFire safety analysis appended to compliance_report.txt")



FIRE SAFETY ROUTE ANALYSIS
Total spaces               : 21
Total doors                : 14
Graph nodes                : 21
Graph edges                : 12
Longest route distance (m) : 3.12
Compliant                  : False
------------------------------------------------------------------------------------------------
Longest Route
------------------------------------------------------------------------------------------------
Start space: B104
Path: B104 -> B101

Exit Spaces
------------------------------------------------------------------------------------------------
1. A102
2. A101
3. B101
4. B102

Safety Issues
------------------------------------------------------------------------------------------------
1. A201: no route to any exit space.
2. B201: no route to any exit space.
3. A205: no route to any exit space.
4. B205: no route to any exit space.
5. A105: no route to any exit space.
6. B105: no route to any exit space.
7. R301: no route to any exit space.
8. A103: no route

## Useful IFC Concepts

### Common Entity Types

- **IfcSpace**: A room, area, or zone in the building
- **IfcWindow**: Windows for light and ventilation  
- **IfcDoor**: Doors, openings, access points
- **IfcWall**: Boundary elements
- **IfcBuildingElement**: General building components
- **IfcElement**: Physical building pieces with properties

### Accessing Properties

```python
# Get all entities of a type
elements = ifc.by_type("IfcWindow")

# Access properties
entity.Name                    # String name
entity.Description             # Text description
entity.ObjectPlacement        # Location/coordinates
entity.Representation         # Geometry data
entity.QuantityInSpace         # Calculate-derived quantities

# Common relationships
entity.HasProperties           # Get properties
entity.BoundedBy               # Spatial boundaries
entity.HostedBy                # Connection relationships
```

### Helpful Methods

```python
# Query by GUID
entity = ifc.by_id(guid)

# Filter by property value
elements = ifc.by_attribute("Name", "Kitchen")

# Get all instances of a type
spaces = ifc.by_type("IfcSpace")

# Check entity type
if space.is_a("IfcSpace"):
    print("This is a space")
```

## Resources for Help

- **IFC Knowledge Base**: https://notebooklm.google.com/notebook/0925c2a1-519b-40a8-aca4-1e832d219f3c
- **IfcOpenShell Documentation**: https://docs.ifcopenshell.org/ifcopenshell-python.html
- **BuildingSmart Standards**: https://www.buildingsmart.org/
- **Catalan Building Code**: https://notebooklm.google.com/notebook/216b245f-0fc1-4063-bdfd-d23b41360b0e

## Next Steps

After completing these exercises, you'll have learned:
- ✓ How to load and explore IFC files
- ✓ How to extract spatial and building data
- ✓ How to validate designs against building codes
- ✓ How to work with doors, windows, and routes

These skills apply to real-world AEC (Architecture, Engineering, Construction) workflows.